In [1]:
from tensorflow.keras.models import model_from_json
import tensorflow as tf
from tensorflow import keras

# Load architecture
with open('model_architecture.json', 'r') as f:
    model_json = f.read()
model = model_from_json(model_json)

# Load weights
model.load_weights('model_weights.weights.h5')

# (Optional) compile for inference/evaluation
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Inspect
model.summary()

Matplotlib is building the font cache; this may take a moment.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 8, 32)      │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 4, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 2, 64)      │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1664)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       106,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,636 (490.77 KB)

 Trainable params: 125,636 (490.77 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.save("cnn_model.h5")

In [13]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

converter.optimizations = [tf.lite.Optimize.DEFAULT]

with open("model.tflite","wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\nb0801\AppData\Local\Temp\tmpvlqx4rfd\assets


INFO:tensorflow:Assets written to: C:\Users\nb0801\AppData\Local\Temp\tmpvlqx4rfd\assets


Saved artifact at 'C:\Users\nb0801\AppData\Local\Temp\tmpvlqx4rfd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 10, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2877404314704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2877404315472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2877405364688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2877405365456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2877405367376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2877405368144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2877405367568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2877405368336: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [15]:
with open("model.tflite", "rb") as f:
    data = f.read()

with open("model_data.cc", "w") as f:
    f.write("const unsigned char model[] = {\n")

    for i, byte in enumerate(data):
        if i % 12 == 0:
            f.write("    ")
        f.write(f"0x{byte:02x}, ")
        if i % 12 == 11:
            f.write("\n")

    f.write("\n};\n")
    f.write(f"\nconst unsigned int model_len = {len(data)};\n")